<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Manual_logic_checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Manual Backtest Logic Checker

This notebook loads the **actual `build_excel_grid_table()` and `run_grid_backtest()` functions from `Grid_trading.ipynb` on the `main` branch**. You only change the manual parameters and OHLC path below, then inspect the output.

This is intentionally a visual/manual checker, not a separate expected-value implementation.

## 1. Load the exact functions from the main notebook

In [ ]:
import ast
import bisect
import heapq
import json
import hashlib

import requests
import numpy as np
import pandas as pd

MAIN_NOTEBOOK_URL = (
    'https://raw.githubusercontent.com/'
    'natdanaiii/Trading/main/Grid_trading.ipynb'
)

response = requests.get(MAIN_NOTEBOOK_URL, timeout=30)
response.raise_for_status()
notebook = response.json()

def load_notebook_function(function_name):
    namespace = {
        'np': np,
        'pd': pd,
        'heapq': heapq,
        'bisect': bisect,
    }

    for cell in notebook['cells']:
        if cell.get('cell_type') != 'code':
            continue

        source = cell.get('source', '')
        if isinstance(source, list):
            source = ''.join(source)

        if f'def {function_name}(' not in source:
            continue

        tree = ast.parse(source)
        for node in tree.body:
            if (
                isinstance(node, ast.FunctionDef)
                and node.name == function_name
            ):
                module = ast.Module(
                    body=[node],
                    type_ignores=[],
                )
                ast.fix_missing_locations(module)
                exec(
                    compile(module, 'Grid_trading.ipynb', 'exec'),
                    namespace,
                )
                function_source = ast.unparse(node)
                return (
                    namespace[function_name],
                    function_source,
                )

    raise RuntimeError(
        f'Function {function_name!r} was not found.'
    )

build_excel_grid_table, grid_source = (
    load_notebook_function('build_excel_grid_table')
)
run_grid_backtest, engine_source = (
    load_notebook_function('run_grid_backtest')
)

source_hash = hashlib.sha256(
    (grid_source + engine_source).encode('utf-8')
).hexdigest()[:16]

print('Loaded functions directly from Grid_trading.ipynb')
print(f'Function source hash: {source_hash}')

## 2. Manual input

Change **only this cell** for your test.

- Strategy parameters use the same names as the real grid function.
- `MANUAL_OHLC` is the 1-minute price path you want to test manually.
- Each row is `(Open, High, Low, Close)`.

In [ ]:
# ============================================================
# MANUAL INPUT — EDIT HERE
# ============================================================

manual_parameters = {
    'capital': 1000.0,
    'ceiling': 130.0,
    'floor': 100.0,
    'gap': 10.0,
    'buy_fee': 0.001,
    'sell_fee': 0.001,
}

# Each row = (Open, High, Low, Close)
MANUAL_OHLC = [
    (125.0, 126.0, 115.0, 118.0),
    (120.0, 131.0, 120.0, 130.0),
]

MANUAL_START_TIME = '2024-01-01 00:00:00+00:00'

## 3. Run the real engine and inspect the result

In [ ]:
df_manual_grid = build_excel_grid_table(
    **manual_parameters
)

manual_times = pd.date_range(
    MANUAL_START_TIME,
    periods=len(MANUAL_OHLC),
    freq='min',
    tz='UTC',
)

df_manual_price = pd.DataFrame({
    'open_time': manual_times,
    'open': [x[0] for x in MANUAL_OHLC],
    'high': [x[1] for x in MANUAL_OHLC],
    'low': [x[2] for x in MANUAL_OHLC],
    'close': [x[3] for x in MANUAL_OHLC],
})

manual_result = run_grid_backtest(
    df_price=df_manual_price,
    grid_table=df_manual_grid,
    initial_capital=manual_parameters['capital'],
)

manual_summary = manual_result['summary']
df_manual_trade_log = manual_result['trade_log']
df_manual_completed = manual_result['completed_trades']
df_manual_equity = manual_result['equity_curve']
df_manual_state = manual_result['grid_state']

print('===== MANUAL BACKTEST LOGIC CHECKER =====')
print()
print('--- INPUT PARAMETERS ---')
print(f'Capital            : {manual_parameters["capital"]:,.6f} USDT')
print(f'Ceiling            : {manual_parameters["ceiling"]:,.6f}')
print(f'Floor              : {manual_parameters["floor"]:,.6f}')
print(f'Gap                : {manual_parameters["gap"]:,.6f}')
print(f'Buy Fee            : {manual_parameters["buy_fee"]:.4%}')
print(f'Sell Fee           : {manual_parameters["sell_fee"]:.4%}')
print(f'Number of Grids    : {len(df_manual_grid)}')
print(f'Capital / Grid     : {df_manual_grid["capital_per_level"].iloc[0]:,.6f} USDT')

print()
print('--- FINAL ENGINE OUTPUT ---')
print(f'Final Cash         : {manual_summary["final_cash"]:,.6f} USDT')
print(f'Final BTC          : {manual_summary["final_btc"]:.12f} BTC')
print(f'Final Equity       : {manual_summary["final_equity"]:,.6f} USDT')
print(f'Realized Profit    : {manual_summary["realized_profit"]:,.6f} USDT')
print(f'Unrealized P&L     : {manual_summary["unrealized_pnl"]:,.6f} USDT')
print(f'Completed Cycles   : {manual_summary["completed_cycles"]}')
print(f'Open Positions     : {manual_summary["open_positions"]}')

print()
print('1) Manual OHLC input')
display(df_manual_price)

print('2) Grid generated by the REAL grid function')
display(
    df_manual_grid[[
        'level',
        'buy_price',
        'sell_price',
        'capital_per_level',
        'gross_base_amount',
        'buy_fee_base',
        'base_amount',
        'gross_sell',
        'sell_fee_quote',
        'net_sell',
        'profit',
    ]]
)

print('3) Trade log generated by the REAL backtest engine')
if len(df_manual_trade_log):
    display(
        df_manual_trade_log[[
            'event_id',
            'time',
            'side',
            'grid_level',
            'price',
            'quote_amount',
            'base_amount',
            'fee_base',
            'fee_quote',
            'cash_movement',
            'grid_cashflow',
            'cash_before',
            'cash_after',
            'btc_before',
            'btc_after',
        ]]
    )
else:
    print('No trade was executed.')

print('4) Completed BUY -> SELL cycles')
if len(df_manual_completed):
    display(
        df_manual_completed[[
            'grid_level',
            'buy_time',
            'sell_time',
            'buy_price',
            'sell_price',
            'cost',
            'base_amount',
            'actual_earn',
            'grid_cashflow',
        ]]
    )
else:
    print('No completed cycle.')

print('5) Portfolio after each manual candle')
display(
    df_manual_equity[[
        'open_time',
        'close',
        'cash',
        'btc',
        'equity',
        'drawdown',
    ]]
)

print('6) Grid positions still open at the end')
manual_open_grids = df_manual_state.loc[
    df_manual_state['holding']
]
if len(manual_open_grids):
    display(
        manual_open_grids[[
            'level',
            'buy_price',
            'sell_price',
            'capital_per_level',
            'base_amount',
            'buy_time',
        ]]
    )
else:
    print('No open grid position.')